# OneRef sur Colab
Runtime recommandé : **GPU L4** (bf16). Exécute les cellules dans l'ordre ; les sorties sont sur Drive donc un run coupé **reprend** là où il s'est arrêté.

In [ ]:
# 1. GPU + profil de modèles
import subprocess, json, os
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
print(gpu)
T4 = "T4" in gpu
# --- T4 : pas de bf16 et 16 Go => checkpoints quantifiés (IDs A VERIFIER sur Hugging Face) ---
T4_MODELS = {"READER_A_MODEL":"Qwen/Qwen3-VL-8B-Instruct-AWQ", "READER_B_MODEL":"OpenGVLab/InternVL3-8B-AWQ", "CRITIC_MODEL":"Qwen/Qwen3-VL-8B-Instruct-AWQ"}
vllm_args = ["--max-model-len","8192","--gpu-memory-utilization","0.9","--trust-remote-code","--enable-auto-tool-choice","--tool-call-parser","hermes"]
if T4:
    vllm_args += ["--dtype","half"]
    for k,v in T4_MODELS.items(): os.environ["ONEREF_"+k]=v
os.environ["ONEREF_VLLM_ARGS"] = json.dumps(vllm_args)

In [ ]:
# 2. Drive + copie du code (uploade le dossier Colab/ dans MyDrive/OneRef/Colab, MMAD dans MyDrive/OneRef/MMAD)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/OneRef"
!rm -rf /content/OneRef_Colab && cp -r "$DRIVE/Colab" /content/OneRef_Colab
%cd /content/OneRef_Colab

In [ ]:
# 3. Installation (vLLM peut remplacer torch : si Colab le demande, redémarre le runtime puis relance depuis la cellule 1)
!pip install -q -r requirements_colab.txt

In [ ]:
# 4. Hugging Face (secret Colab "HF_TOKEN" ; accepter la licence DINOv3 sur le Hub)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# 5. Variables d'environnement du projet
os.environ.update({
    "ONEREF_MMAD_ROOT": f"{DRIVE}/MMAD",
    "ONEREF_MMAD_JSON": f"{DRIVE}/MMAD/mmad.json",
    "ONEREF_OUTPUTS_DIR": f"{DRIVE}/outputs",
    "ONEREF_SPLITS_DIR": f"{DRIVE}/splits",
    "ONEREF_VLLM_AUTOSTART": "true",
    "ONEREF_DEVICE": "cuda",
    "PYTHONPATH": "/content/OneRef_Colab",
})

In [ ]:
# 5b. (OPTIONNEL) Télécharger MMAD directement, au lieu de l'uploader sur Drive. A faire UNE seule fois.
# ~28 Go : il faut la place sur Drive. Alternative : remplacer f"{DRIVE}/MMAD" par "/content/MMAD" (disque local, perdu à la déconnexion).
# Décommente tout le bloc pour l'exécuter.
# from huggingface_hub import snapshot_download
# snapshot_download("jiang-cc/MMAD", repo_type="dataset", local_dir=f"{DRIVE}/MMAD")
# !find "$DRIVE/MMAD" -name "*.zip" -execdir unzip -q -o {} \;
# !git clone --depth 1 https://github.com/jam-cc/MMAD /content/MMAD_code
# !cp "$(find /content/MMAD_code -name mmad.json | head -n 1)" "$DRIVE/MMAD/mmad.json"


In [ ]:
# 6. Smoke test sans GPU
!python -m pytest -q
!python -m inspector.main --split dev --mock --limit 20 --run-name colab_smoke

In [ ]:
# 7. Sonde de coût : 50 images (mesure les unités Colab avant de fixer la taille du test)
!python -m inspector.main --split dev --limit 50 --run-name probe

In [ ]:
# 8. Run dev complet (reprenable : relance la cellule après une coupure)
!python -m inspector.main --split dev --run-name dev01

In [ ]:
# 9. Dernières lignes du log JSONL
!tail -n 3 "$DRIVE"/outputs/dev01/*.jsonl | cut -c1-600

## Test (une seule fois, config figée)
Ne lance **jamais** le split test pour régler des paramètres.

In [ ]:
# !python -m inspector.main --split test --frozen-config --run-name test01